# Filter aggregated classification results

Keep the requested BERTopic and indicator configuration, select and rename the evaluation metrics, and add an average across all models for every account-relations configuration.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
def find_project_root(start_path):
    """Find the project root containing AGENTS.md.

    Args:
        start_path: Directory from which to begin the search.

    Returns:
        Path to the Backbone-Graph project root.
    """
    start_path = Path(start_path).resolve()
    for directory in (start_path, *start_path.parents):
        if (directory / "AGENTS.md").exists():
            return directory
    raise FileNotFoundError("Could not find the Backbone-Graph project root.")

In [6]:
project_root = find_project_root(Path.cwd())
csv_path = (
    project_root
    / "results"
    / "Labeled_Datasets"
    / "step_4_inter_intra_classification"
    / " aggregated"
    / "top_percentage_100_iran_1_uae_venezuela_china_1_cuba_russia_1"
    / "aggregated_classification_sweep_results.csv"
)

classification_df = pd.read_csv(csv_path)

In [7]:
filtered_df = (
    classification_df.loc[
        classification_df["indicators_configuration"].eq("All")
        & classification_df["topic_col"].eq("BERTopic_topic_256"),
        [
            "model",
            "account_relations_configuration",
            "all_authors_auc",
            "all_authors_auc_pr",
        ],
    ]
    .rename(
        columns={
            "all_authors_auc": "roc_auc",
            "all_authors_auc_pr": "pr_auc",
        }
    )
    .reset_index(drop=True)
)

filtered_df

,model,account_relations_configuration,roc_auc,pr_auc
0,logistic_regression,interactions,0.855165,0.634169
1,random_forest,interactions,0.950548,0.860125
2,xgboost,interactions,0.957528,0.856975
3,logistic_regression,similarity,0.922250,0.691871
4,random_forest,similarity,0.992560,0.968559
5,xgboost,similarity,0.995032,0.972331
6,logistic_regression,similarity+interactions,0.912694,0.673213
7,random_forest,similarity+interactions,0.992351,0.968443
8,xgboost,similarity+interactions,0.994159,0.969695
9,logistic_regression,similarity+text_similarity,0.935955,0.766197


In [8]:
metric_columns = ["roc_auc", "pr_auc"]
average_rows = (
    filtered_df.groupby(
        "account_relations_configuration",
        as_index=False,
    )[metric_columns]
    .mean()
)
average_rows.insert(0, "model", "all models average")

results_df = pd.concat(
    [filtered_df, average_rows],
    ignore_index=True,
)

results_df

,model,account_relations_configuration,roc_auc,pr_auc
0,logistic_regression,interactions,0.855165,0.634169
1,random_forest,interactions,0.950548,0.860125
2,xgboost,interactions,0.957528,0.856975
3,logistic_regression,similarity,0.922250,0.691871
4,random_forest,similarity,0.992560,0.968559
5,xgboost,similarity,0.995032,0.972331
6,logistic_regression,similarity+interactions,0.912694,0.673213
7,random_forest,similarity+interactions,0.992351,0.968443
8,xgboost,similarity+interactions,0.994159,0.969695
9,logistic_regression,similarity+text_similarity,0.935955,0.766197


In [10]:
# output_dir = (
#     project_root.parent
#     / "paper_overleaf"
#     / "ComplexNetworksTemplate"
#     / "figs"
# )

output_dir = Path("results/Labeled_Datasets/step_4_inter_intra_classification")

output_dir.mkdir(parents=True, exist_ok=True)
latex_path = output_dir / "filtered_aggregated_classification_results.tex"

results_df.to_latex(
    latex_path,
    index=False,
    float_format="%.3f",
    column_format="llrr",
    escape=True,
)

print(f"Saved LaTeX table: {latex_path}")

Saved LaTeX table: results/Labeled_Datasets/step_4_inter_intra_classification/filtered_aggregated_classification_results.tex
